In [1]:
import pandas as pd
train_model = pd.read_csv("../data/raw/train_model.csv")

In [2]:
sample = train_model.sample(n=50000, random_state=42)

print("train_model shape:", train_model.shape)
print("sample shape:", sample.shape)

print(sample["label"].value_counts())
print(sample["label"].value_counts(normalize=True))

train_model shape: (260864, 6)
sample shape: (50000, 6)
label
0    46972
1     3028
Name: count, dtype: int64
label
0    0.93944
1    0.06056
Name: proportion, dtype: float64


In [3]:
def extract_activity_features(activity_log):
    """
    输入：一行 activity_log 字符串
    输出：这一行对应的行为特征
    
    activity_log 格式：
    item_id:cat_id:brand_id:time_stamp:action_type#item_id:cat_id:brand_id:time_stamp:action_type
    """
    
    # 如果 activity_log 是缺失值，返回全 0 特征
    if pd.isna(activity_log):
        return pd.Series({
            "total_actions": 0,
            "click_cnt": 0,
            "cart_cnt": 0,
            "buy_cnt": 0,
            "fav_cnt": 0,
            "unique_items": 0,
            "unique_cats": 0,
            "unique_brands": 0,
            "unique_days": 0
        })
    
    # 按 # 拆分成多条行为记录
    records = str(activity_log).split("#")
    
    # 用来存储每条行为拆出来的信息
    item_list = []
    cat_list = []
    brand_list = []
    day_list = []
    action_list = []
    
    for record in records:
        # 每条行为理论上应该由 5 个部分组成
        parts = record.split(":")
        
        # 如果不是 5 个部分，说明这一条异常，跳过
        if len(parts) != 5:
            continue
        
        item_id, cat_id, brand_id, time_stamp, action_type = parts
        
        item_list.append(item_id)
        cat_list.append(cat_id)
        brand_list.append(brand_id)
        day_list.append(time_stamp)
        action_list.append(action_type)
    
    # 统计不同类型行为数量
    click_cnt = action_list.count("0")
    cart_cnt = action_list.count("1")
    buy_cnt = action_list.count("2")
    fav_cnt = action_list.count("3")
    
    return pd.Series({
        # 总行为次数
        "total_actions": len(action_list),
        
        # 点击次数
        "click_cnt": click_cnt,
        
        # 加购次数
        "cart_cnt": cart_cnt,
        
        # 购买次数
        "buy_cnt": buy_cnt,
        
        # 收藏次数
        "fav_cnt": fav_cnt,
        
        # 互动过的商品数量
        "unique_items": len(set(item_list)),
        
        # 互动过的品类数量
        "unique_cats": len(set(cat_list)),
        
        # 互动过的品牌数量
        "unique_brands": len(set(brand_list)),
        
        # 活跃天数
        "unique_days": len(set(day_list))
    })

In [4]:
# 对 sample 的 activity_log 进行特征提取
activity_features = sample["activity_log"].apply(extract_activity_features)

# 把提取出来的特征和原始字段合并
sample_features = pd.concat(
    [
        sample[["user_id", "age_range", "gender", "merchant_id", "label"]].reset_index(drop=True),
        activity_features.reset_index(drop=True)
    ],
    axis=1
)

display(sample_features.head())
print(sample_features.shape)

,user_id,age_range,gender,merchant_id,label,total_actions,click_cnt,cart_cnt,buy_cnt,fav_cnt,unique_items,unique_cats,unique_brands,unique_days
0,399620,3.0,1.0,310,0,9,8,0,1,0,4,3,1,1
1,183656,2.0,1.0,3129,0,3,2,0,1,0,2,1,1,1
2,214005,0.0,0.0,3609,0,12,9,0,1,2,2,1,1,5
3,76778,0.0,1.0,2824,0,3,2,0,1,0,2,1,1,1
4,258043,0.0,0.0,4760,1,3,2,0,1,0,1,1,1,1


(50000, 14)


In [6]:
# 防止除以 0 的函数
def safe_divide(a, b):
    return a / (b + 1e-6)

# 购买 / 点击，表示点击后的购买转化倾向
sample_features["buy_click_rate"] = safe_divide(
    sample_features["buy_cnt"], 
    sample_features["click_cnt"]
)

# 加购 / 点击，表示用户对该商家的潜在购买意愿
sample_features["cart_click_rate"] = safe_divide(
    sample_features["cart_cnt"], 
    sample_features["click_cnt"]
)

# 收藏 / 点击，表示用户对该商家的兴趣强度
sample_features["fav_click_rate"] = safe_divide(
    sample_features["fav_cnt"], 
    sample_features["click_cnt"]
)

# 购买 / 总行为，表示用户行为中的购买占比
sample_features["buy_action_rate"] = safe_divide(
    sample_features["buy_cnt"], 
    sample_features["total_actions"]
)

display(sample_features.head())
sample_features.to_csv("../data/raw/train_features_sample_50000.csv", index=False)

,user_id,age_range,gender,merchant_id,label,total_actions,click_cnt,cart_cnt,buy_cnt,fav_cnt,unique_items,unique_cats,unique_brands,unique_days,buy_click_rate,cart_click_rate,fav_click_rate,buy_action_rate
0,399620,3.0,1.0,310,0,9,8,0,1,0,4,3,1,1,0.125000,0.0,0.000000,0.111111
1,183656,2.0,1.0,3129,0,3,2,0,1,0,2,1,1,1,0.500000,0.0,0.000000,0.333333
2,214005,0.0,0.0,3609,0,12,9,0,1,2,2,1,1,5,0.111111,0.0,0.222222,0.083333
3,76778,0.0,1.0,2824,0,3,2,0,1,0,2,1,1,1,0.500000,0.0,0.000000,0.333333
4,258043,0.0,0.0,4760,1,3,2,0,1,0,1,1,1,1,0.500000,0.0,0.000000,0.333333
